# Subseasonal to Seasonal (S2S) – Neptune, Persistence and Climatology

This notebook provides tools for the analysis and comparison of subseasonal to seasonal (S2S) forecasts using machine learning emulators. Key features include:

- **Loading and visualization** of S2S forecast and reanalysis datasets.
- **Calculation and comparison** of evaluation metrics (e.g., RMSE, ACC, CRPS) for different models and observations.
- **Temporal and spatial** analysis of model performance.
- **Visualization of maps**, time series, and statistical distributions.
- **Support for multiple datasets** and atmospheric/oceanic variables.
- **Interactive exploration** of results for model validation and benchmarking.

This notebook is designed to facilitate model comparison, forecast validation, and in-depth performance analysis over weekly to seasonal timescales.

In [ ]:
# NOTEBOOK PARAMETERS
selected_model = 'neptune_1'
ensemble_size = 8
years = [2019,2020,2021]

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import warnings
import munch
import toml
import sys
import os

warnings.filterwarnings('ignore')
sys.path.append('library')
from atmocean.metrics import compute_and_store_metrics, get_persistence_ds
from atmocean.utils import clean_forecasts, remove_ocean_variables_suffix
from atmocean.macros import ALL_DRIVERS, MODEL_NAMES
from atmocean.visualize import *

os.system('hostname')
%load_ext autoreload
%autoreload 2

In [ ]:
forecast_horizon = 60

start_date = '01-01'
end_date = '12-31'
years = [2019,2020,2021]

lead_times = [0, 10, 20, 30, 40, 50, 60]

fsystem = 'ocean'
main_dir = 'PATH-TO-MAIN-DIRECTORY'
data_dir = os.path.join(main_dir, 'data')
forecast_dir = os.path.join(data_dir, 'forecasts')
metrics_src = os.path.join(data_dir, 'evaluation_metrics')

persistence_src = os.path.join(forecast_dir, f'persistence-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
climatology_src = os.path.join(forecast_dir, f'climatology-from_1993_to_2017.zarr')

if ensemble_size is None:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}/statistical_evaluation')
else:
    forecasts_srcs = [
        os.path.join(forecast_dir, f'{selected_model}-from_{year}-{start_date}_to_{year}-{end_date}-ENS-{ensemble_size:02d}.zarr')
        for year in years
    ]
    out_img_dir = os.path.join(main_dir, f'validation/img/{selected_model}/s2s-{years[0]}-{years[-1]}-ENS-{ensemble_size:02d}/statistical_evaluation')

if '025' in selected_model:
    persistence_src = persistence_src+'-025'
    climatology_src = climatology_src+'-025'

os.makedirs(out_img_dir, exist_ok=True)
plot_kwargs = dict(dpi=100, bbox_inches='tight')
save_figures = True

In [ ]:
# get all the dates
dates = pd.date_range(f'{years[0]}-{start_date}', f'{years[-1]}-{end_date}', freq='1D')
year = f'{dates[0].year}'
sdate = f'{dates[0].year}-{dates[0].month:02d}-{dates[0].day:02d}'
edate = f'{dates[-1].year}-{dates[-1].month:02d}-{dates[-1].day:02d}'
sdate, edate

In [ ]:
fields_info = munch.munchify(toml.load(os.path.join(data_dir, 'specs/ocean_fields.toml')))
fields_info

In [ ]:
season = 'annual'

if ensemble_size is None:
    forecast_crps_src = os.path.join(metrics_src, f'crps', f'{selected_model}-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
else:
    forecast_crps_src = os.path.join(metrics_src, f'crps', f'{selected_model}-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}-ENS-{ensemble_size:02d}.zarr')

persistence_crps_src = os.path.join(metrics_src, f'crps', f'persistence-{season}-from_{years[0]}-{start_date}_to_{years[-1]}-{end_date}.zarr')
climatology_crps_src = os.path.join(metrics_src, f'crps', f'climatology-{season}-from_1993_to_2017.zarr')

In [ ]:
# open forecasts
forecasts_ds = xr.open_mfdataset(forecasts_srcs, chunks={}, parallel=True, engine='zarr', preprocess=clean_forecasts)

# open the climatology dataset
climatology_ds = xr.open_zarr(climatology_src, chunks={})

# open the persistence dataset
persistence_ds = get_persistence_ds(persistence_src, forecasts_ds.isel(ensemble_member=0, missing_dims='ignore'))

In [ ]:
# realign the prediction_timedelta range
forecasts_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)
climatology_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)
persistence_ds['prediction_timedelta'] = pd.timedelta_range('0 day', periods=forecast_horizon+1)

In [ ]:
forecasts_ds = forecasts_ds.isel(level=0, prediction_timedelta=lead_times, missing_dims='ignore')
climatology_ds = climatology_ds.isel(level=0, prediction_timedelta=lead_times, missing_dims='ignore')
persistence_ds = persistence_ds.isel(level=0, prediction_timedelta=lead_times, missing_dims='ignore')

In [ ]:
# get only the drivers present in the dataset
all_fields = list(forecasts_ds.data_vars.keys())
drivers = ALL_DRIVERS
drivers['output']['atmos']['fields_3d'] = [field for field in drivers['output']['atmos']['fields_3d'] if field in all_fields]
drivers['output']['atmos']['fields_2d'] = [field for field in drivers['output']['atmos']['fields_2d'] if field in all_fields]
drivers['output']['ocean']['fields_3d'] = [field for field in drivers['output']['ocean']['fields_3d'] if field in all_fields]
drivers['output']['ocean']['fields_2d'] = [field for field in drivers['output']['ocean']['fields_2d'] if field in all_fields]

In [ ]:
def create_yearly_climatology(climatology_ds: xr.Dataset, years):
    clm = []
    for year in years:
        dates = pd.date_range(f'{year}-01-01', f'{year}-12-31', freq='1D')
        tmp = climatology_ds.copy()
        if len(dates) == 365:
            tmp = tmp.sel(time=~((tmp.time.dt.month == 2) & (tmp.time.dt.day == 29)))
        tmp['time'] = dates
        clm.append(tmp)
    clms = xr.concat(clm, dim='time')
    return clms

In [ ]:
# align climatology with the ground truth coming from 
# the persistence, as it contains the real values of 
# ground truth at `prediction_timedelta` = 0
climatology_ds = create_yearly_climatology(climatology_ds, years)
climatology_ds = xr.concat([
    persistence_ds.isel(prediction_timedelta=slice(0,1)), # correct ground truth
    climatology_ds.isel(prediction_timedelta=slice(1,None))
    ], dim='prediction_timedelta'
)

In [ ]:
valid_start_date = pd.to_datetime(f'{years[0]}-{start_date}') + pd.DateOffset(days=forecast_horizon)
valid_end_date = pd.to_datetime(f'{years[-1]}-{end_date}')
valid_start_date, valid_end_date

In [ ]:
from atmocean.losses import CRPS

from tqdm import tqdm
import pandas as pd
import xarray as xr
import numpy as np
import torch
import os


def compute_crps_group(
        forecasts_paired,
        initial_conditions,
        field_list,
        device: str = 'cpu',
    ):
    fc = forecasts_paired[field_list].to_array(dim='field')
    ic = initial_conditions[field_list].to_array(dim='field')

    fc_t = torch.as_tensor(fc.values, device=device)
    ic_t = torch.as_tensor(ic.values, device=device)

    ens_dim = fc.dims.index('ensemble_member')
    mu = torch.mean(fc_t, dim=ens_dim)
    std = torch.std(fc_t, dim=ens_dim) + 1e-7
    y = ic_t.unsqueeze(2).expand_as(mu)

    crps = CRPS(y=y, mu=mu, std_dev=std).cpu().numpy()

    dims_list = list(fc.dims)
    del dims_list[ens_dim]
    out_dims = tuple(dims_list)

    assert len(out_dims) == crps.ndim, f"Mismatch dims {out_dims} vs shape {crps.shape}"

    del fc_t, ic_t, mu, std, y
    if device != 'cpu':
        torch.cuda.empty_cache()

    return crps, out_dims

def compute_and_store_crps_metrics(
        ds: xr.Dataset,
        dst,
        tmp_dir,
        fields_3d,
        fields_2d,
        valid_start_date,
        valid_end_date,
        lead_times,
        batch_size: int = 30,
    ):
    if os.path.exists(dst):
        return xr.open_zarr(dst)
    all_fields = fields_3d + fields_2d
    valid_dates = pd.date_range(valid_start_date, valid_end_date, freq='1D')
    date_offsets = pd.to_timedelta(lead_times, unit='days')
    n_lead = len(lead_times)
    n_batches = int(np.ceil(len(valid_dates) / batch_size))
    tmp_dst = os.path.join(tmp_dir, '')
    first_write = not __import__('os').path.exists(tmp_dst)

    for b in tqdm(range(n_batches), desc='batch'):
        batch_dates = valid_dates[b * batch_size: (b + 1) * batch_size]
        n_valid_b = len(batch_dates)

        fc_dates = batch_dates.values[:, None] - date_offsets.values[None, :]  # (n_valid_b, n_lead)
        unique_init_dates = pd.DatetimeIndex(np.unique(fc_dates.ravel()))

        ds_sel = ds[all_fields].sel(time=unique_init_dates).load()
        time_index = pd.Index(ds_sel.time.values)

        time_pos = time_index.get_indexer(fc_dates.ravel()).reshape(n_valid_b, n_lead)
        lead_pos = np.broadcast_to(np.arange(n_lead), (n_valid_b, n_lead))

        time_idx = xr.DataArray(time_pos, dims=('valid_time', 'lead'))
        lead_idx = xr.DataArray(lead_pos, dims=('valid_time', 'lead'))

        forecasts_paired = ds_sel.isel(time=time_idx, prediction_timedelta=lead_idx)

        ic_pos = time_index.get_indexer(batch_dates)
        initial_conditions = ds_sel.isel(
            time=xr.DataArray(ic_pos, dims='valid_time'),
            prediction_timedelta=0,
            ensemble_member=0,
        )

        crps_3d, dims_3d = compute_crps_group(forecasts_paired, initial_conditions, fields_3d)
        crps_2d, dims_2d = compute_crps_group(forecasts_paired, initial_conditions, fields_2d)

        spatial_coords_3d = {d: forecasts_paired[fields_3d[0]].coords[d] for d in dims_3d if d not in ('field', 'valid_time', 'lead')}
        spatial_coords_2d = {d: forecasts_paired[fields_2d[0]].coords[d] for d in dims_2d if d not in ('field', 'valid_time', 'lead')}

        batch_result = xr.Dataset(
            {
                **{f'crps_{f}': (dims_3d[1:], crps_3d[i]) for i, f in enumerate(fields_3d)},
                **{f'crps_{f}': (dims_2d[1:], crps_2d[i]) for i, f in enumerate(fields_2d)},
            },
            coords={'valid_time': batch_dates, 'lead': lead_times, **spatial_coords_3d, **spatial_coords_2d},
        )

        if first_write:
            batch_result.to_zarr(tmp_dst, mode='w', zarr_format=2)
            first_write = False
        else:
            batch_result.to_zarr(tmp_dst, mode='a', append_dim='valid_time', zarr_format=2)

        del ds_sel, forecasts_paired, initial_conditions, batch_result
    
    # pre-process the zarr dataset and store into the final directory
    crps_ds = xr.open_zarr(tmp_dst)
    crps_ds = crps_ds.rename_dims({'valid_time':'time', 'lead':'prediction_timedelta'}).rename_vars({'valid_time':'time', 'lead':'prediction_timedelta'})
    rename_vars_dict = {f'crps_{field}':field for field in all_fields}
    crps_ds = crps_ds.rename_vars(rename_vars_dict)
    crps_ds['prediction_timedelta'] = pd.to_timedelta(crps_ds['prediction_timedelta'].values, unit='days')
    crps_ds = crps_ds.mean(dim='time', skipna=True)
    try:
        crps_ds.to_zarr(dst, zarr_format=2)
    except:
        print('Error occurred during dataset storing. Store by yourself')
        return crps_ds

In [ ]:
tmp_dir = 'data/tmp'
os.makedirs(tmp_dir, exist_ok=True)
tmp_fcs_dst = os.path.join(tmp_dir, 'forecasts_crps_tmp.zarr')
tmp_prs_dst = os.path.join(tmp_dir, 'persistence_crps_tmp.zarr')
tmp_clm_dst = os.path.join(tmp_dir, 'climatology_crps_tmp.zarr')

In [ ]:
fields_3d = ['thetao_oras', 'so_oras', 'uo_oras', 'vo_oras']
fields_2d = ['zos_oras', 'mlotst_oras', 'siconc_oras', 'sithick_oras']
all_fields = fields_3d + fields_2d

In [ ]:
# compute the crps and store it to zarr
forecasts_crps = compute_and_store_crps_metrics(
    ds=forecasts_ds,
    dst=forecast_crps_src,
    tmp_dir=tmp_fcs_dst,
    fields_3d=fields_3d,
    fields_2d=fields_2d,
    valid_start_date=valid_start_date,
    valid_end_date=valid_end_date,
    lead_times=lead_times,
    batch_size=30,
)
persistence_crps = compute_and_store_crps_metrics(
    ds=persistence_ds,
    dst=persistence_crps_src,
    tmp_dir=tmp_prs_dst,
    fields_3d=fields_3d,
    fields_2d=fields_2d,
    valid_start_date=valid_start_date,
    valid_end_date=valid_end_date,
    lead_times=lead_times,
    batch_size=30,
)
climatology_crps = compute_and_store_crps_metrics(
    ds=climatology_ds,
    dst=climatology_crps_src,
    tmp_dir=tmp_clm_dst,
    fields_3d=fields_3d,
    fields_2d=fields_2d,
    valid_start_date=valid_start_date,
    valid_end_date=valid_end_date,
    lead_times=lead_times,
    batch_size=30,
)

In [ ]:
# get the fields
fields_3d_ = drivers['output']['atmos']['fields_3d'] + drivers['output']['ocean']['fields_3d']
fields_2d_ = drivers['output']['atmos']['fields_2d'] + drivers['output']['ocean']['fields_2d']
# remove the suffix
fields_3d, fields_2d = [], []
for field in fields_3d_:
    field_rename = '_'.join(field.split('_')[:-1])
    fields_3d.append(field_rename)
for field in fields_2d_:
    field_rename = '_'.join(field.split('_')[:-1])
    fields_2d.append(field_rename)
# add ocean fields
ice_fields_2d = ['siconc', 'sithick']
other_fields_2d = [field for field in fields_2d if field not in ice_fields_2d]
# rename fields info keys
fields_info_new = fields_info.copy()
for key,value in fields_info.items():
    new_key = '_'.join(key.split('_')[:-1])
    fields_info_new[new_key] = value
    del fields_info_new[key]
fields_info = fields_info_new.copy()

In [ ]:
persistence_crps['prediction_timedelta'] = pd.to_timedelta(persistence_crps['prediction_timedelta'].values, unit='days')
climatology_crps['prediction_timedelta'] = pd.to_timedelta(climatology_crps['prediction_timedelta'].values, unit='days')
# persistence_crps = persistence_crps.isel(prediction_timedelta=lead_times)
climatology_crps = climatology_crps.isel(prediction_timedelta=lead_times)

In [ ]:
metrics_dict = {
    'crps': {
        'neptune': remove_ocean_variables_suffix(forecasts_crps),
        'pers': remove_ocean_variables_suffix(persistence_crps),
        'clm': remove_ocean_variables_suffix(climatology_crps),
    },
}

In [ ]:
model_names = {
    'neptune': 'Neptune',
    'pers': 'Persistence',
    'clm': 'Climatology',
}

In [ ]:
def parse_ids(ids: str):
    ri, ci = ids.split(',')
    if ':' in ri:
        ri = ri.split(':')
        ri = slice(int(ri[0]), int(ri[1]))
    else: ri = int(ri)
    if ':' in ci:
        ci = ci.split(':')
        ci = slice(int(ci[0]), int(ci[1]))
    else: ci = int(ci)
    return ri, ci

In [ ]:
fields_info['mlotst']['long_name'] = 'Mixed Layer Depth'

In [ ]:
metric_name = 'crps'
grid_spec_args = dict(num_figures = (2, 6), positions = [
    '0,0:2', '0,2:4', '0,4:6', '1,0:2', '1,2:4', '1,4:6',
])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
per_variable_vmax = {
    'thetao': 4.0, 'so': 1.5, 'uo': 0.6, 'vo': 0.6, 'zos': 0.5, 'mlotst': 100.0, 'siconc': 0.7, 'sithick': 1.0, 
}
fields = fields_3d + fields_2d[:2]

fig, axes = plt.subplots(1, 1, figsize=(12,6))
axes.axis('off')

# initialize grid spec
gs = GridSpec(*grid_spec_args['num_figures'], figure=fig)
axs = []

for i,field in tqdm(enumerate(fields), total=len(fields)):
    metric = metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=-1, depth=0, level=0, missing_dims='ignore')[field].values
    cmap, norm, levels = get_colormap_from_cmap('Spectral_r', vmin=0.0, vmax=per_variable_vmax[field])
    grid_pos = grid_spec_args['positions'][i]
    im, ax = plot_spatial_metric_on_globe(
        metric=metric, grid=gs[parse_ids(grid_pos)], title=fields_info[field]['long_name'].capitalize(),
        fig=fig, lons=lons, lats=lats, central_longitude=180,
        cmap=cmap, norm=norm, levels=levels, left_labels=True if (i==0 or i==3) else False, bottom_labels=True
    )
    axs.append(ax)
    # single colorbar
    cbar_label = fields_info[field]["units"]
    if cbar_label == 'degrees_C': cbar_label = '°C'
    kwargs = dict(
        ax=ax, orientation='horizontal', shrink=0.9,
    )
    kwargs['label'] = cbar_label
    kwargs['ticks'] = np.linspace(0, per_variable_vmax[field], 5)
    cbar = fig.colorbar(im, **kwargs)
# adjust the plot
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=-0.1, hspace=0.3)
outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_lead_time_60_two_rows.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
metric_name = 'crps'
grid_spec_args = dict(num_figures = (1, 12), positions = [
    '0,0:2', '0,2:4', '0,4:6', '0,6:8', '0,8:10', '0,10:12',
])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
per_variable_vmax = {
    'thetao': 4.0, 'so': 1.5, 'uo': 0.6, 'vo': 0.6, 'zos': 0.5, 'mlotst': 100.0, 'siconc': 0.7, 'sithick': 1.0, 
}
fields = fields_3d + fields_2d[:2]

fig, axes = plt.subplots(1, 1, figsize=(24,3))
fig.suptitle(f'{metric_name.upper()} at t+60 lead time', fontsize=16, y=1.1)
axes.axis('off')

# initialize grid spec
gs = GridSpec(*grid_spec_args['num_figures'], figure=fig)
axs = []

for i,field in tqdm(enumerate(fields), total=len(fields)):
    metric = metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=-1, depth=0, level=0, missing_dims='ignore')[field].values
    # cmap, norm, levels = get_sst_bias_colormap(0.0, per_variable_vmax[field])
    cmap, norm, levels = get_colormap_from_cmap('Spectral_r', vmin=0.0, vmax=per_variable_vmax[field])
    grid_pos = grid_spec_args['positions'][i]
    im, ax = plot_spatial_metric_on_globe(
        metric=metric, grid=gs[parse_ids(grid_pos)], title=fields_info[field]['long_name'].capitalize(),
        fig=fig, lons=lons, lats=lats, central_longitude=180,
        cmap=cmap, norm=norm, levels=levels, left_labels = True if i == 0 else False, bottom_labels = True
    )
    axs.append(ax)
    # single colorbar
    cbar_label = fields_info[field]["units"]
    if cbar_label == 'degrees_C': cbar_label = '°C'
    kwargs = dict(
        ax=ax, orientation='horizontal', shrink=0.9,
    )
    kwargs['label'] = cbar_label
    kwargs['ticks'] = np.linspace(0, per_variable_vmax[field], 5)
    cbar = fig.colorbar(im, **kwargs)
# adjust the plot
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.1, top=0.9, wspace=0.0, hspace=0.5)
outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_lead_time_60.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

# Continuous Ranked Probability Score

In [ ]:
metric_name = 'crps'

In [ ]:
grid_spec_args = dict(num_figures = (2, 6), positions = ['0,0:2', '0,2:4', '0,4:6', '1,0:2', '1,2:4', '1,4:6'])
lats, lons = forecasts_ds.latitude.data, forecasts_ds.longitude.data
per_variable_vmax = {
    'thetao': 4.0, 'so': 1.5, 'uo': 0.6, 'vo': 0.6, 'zos': 0.5, 'mlotst': 200.0, 'siconc': 0.7, 'sithick': 1.0, 
}
for field in tqdm(fields_3d + fields_2d):
    outfile = os.path.join(out_img_dir, f'{metric_name}_spatial_{field}.png') if save_figures else None
    metric_values = [
        metrics_dict[metric_name]['neptune'].isel(prediction_timedelta=i+1, depth=0, level=0, missing_dims='ignore')[field].values
        for i,t in enumerate(lead_times[1:])
    ]
    cmap, norm, levels = get_sst_bias_colormap(0.0, per_variable_vmax[field])
    lead_time_labels = [f't+{t}' for t in lead_times[1:]]
    plot_metrics_on_robinson_globe(
        metric_values=metric_values, 
        model_names=lead_time_labels, 
        grid_spec_args=grid_spec_args, 
        central_longitude=180, 
        metric_name=metric_name, 
        field=field, 
        fields_info=fields_info, 
        lats=lats, 
        lons=lons, 
        cmap=cmap, 
        norm=norm, 
        levels=levels, 
        start_year=start_date, 
        end_year=end_date, 
        outfile=outfile, 
        use_units=True, 
    )

In [ ]:
polar = 'north'

model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=1e-2, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=model_names[model1], 
    polar=polar, 
    cmap='jet', 
    divide_by=1, 
    outfile=outfile
)

model1 = 'neptune'
model2 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=None, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=f'({model_names[model1]} - {model_names[model2]})', 
    polar=polar, 
    cmap='RdBu_r', 
    divide_by=2, 
    outfile=outfile
)

model1 = 'neptune'
model2 = 'clm'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=None, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=f'({model_names[model1]} - {model_names[model2]})', 
    polar=polar, 
    cmap='RdBu_r', 
    divide_by=2, 
    outfile=outfile
)

In [ ]:
polar = 'south'

model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=1e-2, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=model_names[model1], 
    polar=polar, 
    cmap='jet', 
    divide_by=1, 
    outfile=outfile
)

model1 = 'neptune'
model2 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=None, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=f'({model_names[model1]} - {model_names[model2]})', 
    polar=polar, 
    cmap='RdBu_r', 
    divide_by=2, 
    outfile=outfile
)

model1 = 'neptune'
model2 = 'clm'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_vs_{model2}_over_{polar}_pole.png') if save_figures else None
polar_plot_ice_fields(
    ds=metrics_dict[metric_name][model1] - metrics_dict[metric_name][model2], 
    lead_times=lead_times[1:], 
    fields=ice_fields_2d, 
    fields_info=fields_info, 
    remove_below_x=None, 
    sdate=sdate, 
    edate=edate, 
    metric_name=metric_name, 
    model_name=f'({model_names[model1]} - {model_names[model2]})', 
    polar=polar, 
    cmap='RdBu_r', 
    divide_by=2, 
    outfile=outfile
)

In [ ]:
outfile = os.path.join(out_img_dir, f'{metric_name}_surface_all_models_compared.png') if save_figures else None
metrics_list = [
    metrics_dict[metric_name]['neptune'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['pers'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['clm'].mean(dim=['latitude','longitude']), 
]
labels = [
    model_names['neptune'], 
    model_names['pers'], 
    model_names['clm'], 
]
colors = [
    'red', 
    'orange', 
    'purple', 
]
surface_plots_multiple_models(
    metrics_list=metrics_list, 
    labels=labels, 
    colors=colors, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    sdate=sdate, 
    edate=edate, 
    fields_info=fields_info, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    metric_name=metric_name, 
    outfile=outfile, 
)

In [ ]:
outfile = os.path.join(out_img_dir, f'{metric_name}_vertical_profile_all_models_compared.png') if save_figures else None
metrics_list = [
    metrics_dict[metric_name]['neptune'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['pers'].mean(dim=['latitude','longitude']), 
    metrics_dict[metric_name]['clm'].mean(dim=['latitude','longitude']), 
]
labels = [
    model_names['neptune'], 
    model_names['pers'], 
    model_names['clm'], 
]
colors = [
    'red', 
    'orange', 
    'purple', 
]
vertical_plots_multiple_models(
    metrics_list=metrics_list, 
    labels=labels, 
    colors=colors, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    vert_levels=drivers['depths'], 
    fields_info=fields_info, 
    sdate=sdate, 
    edate=edate, 
    fields_3d=fields_3d, 
    metric_name=metric_name, 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_scorecard_{model1}.png') if save_figures else None
metrics_scorecard_by_lead_times(
    metrics_ds=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    model=model_names[model1], 
    cmap='PuBu', 
    outfile=outfile, 
)

In [ ]:
model1 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_scorecard_{model1}.png') if save_figures else None
metrics_scorecard_by_lead_times(
    metrics_ds=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    model=model_names[model1], 
    cmap='PuBu', 
    outfile=outfile, 
)

In [ ]:
model1 = 'clm'
outfile = os.path.join(out_img_dir, f'{metric_name}_scorecard_{model1}.png') if save_figures else None
metrics_scorecard_by_lead_times(
    metrics_ds=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    model=model_names[model1], 
    cmap='PuBu', 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
model2 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_difference_{model1}_vs_{model2}.png') if save_figures else None
metrics_difference_scorecard_by_lead_times(
    metrics_model1=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metrics_model2=metrics_dict[metric_name][model2].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    model1=model_names[model1], 
    model2=model_names[model2], 
    cmap='RdBu_r', 
    normalize=False, 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
model2 = 'clm'
outfile = os.path.join(out_img_dir, f'{metric_name}_difference_{model1}_vs_{model2}.png') if save_figures else None
metrics_difference_scorecard_by_lead_times(
    metrics_model1=metrics_dict[metric_name][model1].mean(dim=['latitude','longitude']), 
    metrics_model2=metrics_dict[metric_name][model2].mean(dim=['latitude','longitude']), 
    metric_name=metric_name, 
    fields_info=fields_info, 
    fsystem=fsystem, 
    sdate=sdate, 
    edate=edate, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    vert_levels=drivers['depths'], 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    model1=model_names[model1], 
    model2=model_names[model2], 
    cmap='RdBu_r', 
    normalize=False, 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}_spatial.png') if save_figures else None
plot_spatial_metric(
    metric_spatial=metrics_dict[metric_name][model1], 
    metric_name=metric_name, 
    model=model_names[model1], 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    fields_info=fields_info, 
    lead_times=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    sdate=sdate, 
    edate=edate, 
    outfile=outfile, 
    # divide_by=5, 
    cmap='Spectral_r', 
)

In [ ]:
model1 = 'neptune'
model2 = 'pers'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}-{model2}_spatial_difference.png') if save_figures else None
plot_spatial_metrics_difference(
    metric_model1=metrics_dict[metric_name][model1], 
    metric_model2=metrics_dict[metric_name][model2], 
    metric_name=metric_name, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    fields_info=fields_info, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    model1=model_names[model1], 
    model2=model_names[model2], 
    sdate=sdate, 
    edate=edate, 
    cmap='RdBu_r', 
    outfile=outfile, 
)

In [ ]:
model1 = 'neptune'
model2 = 'clm'
outfile = os.path.join(out_img_dir, f'{metric_name}_{model1}-{model2}_spatial_difference.png') if save_figures else None
plot_spatial_metrics_difference(
    metric_model1=metrics_dict[metric_name][model1], 
    metric_model2=metrics_dict[metric_name][model2], 
    metric_name=metric_name, 
    fields_3d=fields_3d, 
    fields_2d=fields_2d, 
    fields_info=fields_info, 
    lead_times_ids=[i+1 for i in range(len(lead_times[1:]))], 
    lead_times_labels=lead_times[1:], 
    model1=model_names[model1], 
    model2=model_names[model2], 
    sdate=sdate, 
    edate=edate, 
    cmap='RdBu_r', 
    outfile=outfile, 
)